# Augmented CNN
Architecture 3: same model as Architecture 2 but with data augmentation.

Two WandB runs:
- Run 1: no augmentation (shows overfitting baseline for this architecture)
- Run 2: with augmentation (shows improvement from augmentation)

In [ ]:
pip install torch torchvision torchaudio pandas numpy matplotlib scikit-learn wandb

# Colab & Kaggle Setup
Run this section only in Google Colab. Paste your Kaggle access token below.

In [ ]:
!mkdir -p ~/.kaggle

!echo "PASTE_YOUR_TOKEN_HERE" > ~/.kaggle/access_token

!chmod 600 ~/.kaggle/access_token

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/

# WandB Login

In [ ]:
import wandb
wandb.login()

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split

import wandb

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load Data

In [ ]:
DATA_DIR = "data"  # Colab path after Kaggle download
# DATA_DIR = "../data"  # uncomment if running locally

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

print(train_df.shape)
train_df.head()

# Train / Validation Split

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["emotion"]
)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

# Transforms

Two separate pipelines:
- `train_transform` — applies random augmentations so the model sees a different version of each image every epoch
- `val_transform` — no augmentation, always the clean original image for fair evaluation

Augmentations used:
- `RandomHorizontalFlip` — a flipped face has the same emotion, doubles effective variety
- `RandomRotation(10)` — small tilts, faces are rarely perfectly upright
- `RandomCrop(48, padding=4)` — pads image by 4px then crops back to 48x48, teaching position invariance
- `ToTensor` — converts PIL Image [0,255] to tensor [0,1] automatically

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomCrop(48, padding=4),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Dataset

The Dataset class now accepts a `transform` parameter.
When creating the train dataset we pass `train_transform`, for validation we pass `val_transform`.
The image is converted to a PIL Image first so torchvision transforms can work on it.

In [ ]:
class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        pixels = np.array(row["pixels"].split(), dtype=np.uint8)
        image = pixels.reshape(48, 48)
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        label = int(row["emotion"])
        label = torch.tensor(label, dtype=torch.long)

        return image, label

# Model

Same architecture as Architecture 2 (BatchNorm + Dropout).
Keeping the model identical isolates augmentation as the only variable between Run 1 and Run 2.

In [ ]:
class AugmentedCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Sanity Checks
Verify forward and backward passes work before full training.

In [ ]:
temp_dataset = FERDataset(train_data, transform=val_transform)
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=True)

model = AugmentedCNN().to(device)
images, labels = next(iter(temp_loader))
images = images.to(device)

outputs = model(images)
print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, labels = next(iter(temp_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

---
# Run 1: No Augmentation
Same model, but both train and val datasets use `val_transform` (no random augmentations).
Expected result: overfitting — train accuracy will climb, val accuracy will plateau.

In [ ]:
train_dataset_noaug = FERDataset(train_data, transform=val_transform)
val_dataset_noaug = FERDataset(val_data, transform=val_transform)

train_loader_noaug = DataLoader(train_dataset_noaug, batch_size=64, shuffle=True)
val_loader_noaug = DataLoader(val_dataset_noaug, batch_size=64, shuffle=False)

In [ ]:
run = wandb.init(
    entity="nestor_dzadzamia-free-university-of-tbilisi-",
    project="facial-expression-recognition",
    name="augmented-cnn-no-aug",
    config={
        "architecture": "AugmentedCNN",
        "dataset": "FER2013",
        "epochs": 30,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "num_conv_layers": 3,
        "filters": "32->64->128",
        "fc_units": 256,
        "dropout": 0.5,
        "batch_norm": True,
        "augmentation": False,
    }
)

In [ ]:
NUM_EPOCHS = run.config.epochs

model = AugmentedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_noaug:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader_noaug:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

In [ ]:
wandb.finish()

---
# Run 2: With Augmentation
Same model, but train dataset now uses `train_transform` (random flip, rotation, crop).
Val dataset still uses `val_transform` — always clean images for fair evaluation.
Expected result: smaller train/val gap, better generalization.

In [ ]:
train_dataset_aug = FERDataset(train_data, transform=train_transform)
val_dataset_aug = FERDataset(val_data, transform=val_transform)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=64, shuffle=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=64, shuffle=False)

In [ ]:
run = wandb.init(
    entity="nestor_dzadzamia-free-university-of-tbilisi-",
    project="facial-expression-recognition",
    name="augmented-cnn-with-aug",
    config={
        "architecture": "AugmentedCNN",
        "dataset": "FER2013",
        "epochs": 30,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "num_conv_layers": 3,
        "filters": "32->64->128",
        "fc_units": 256,
        "dropout": 0.5,
        "batch_norm": True,
        "augmentation": True,
    }
)

In [ ]:
NUM_EPOCHS = run.config.epochs

model = AugmentedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader_aug:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

In [ ]:
wandb.finish()